### https://www.kaggle.com/competitions/drawing-with-llms

In [14]:
import kagglehub
import pandas as pd

In [15]:
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

svg_constraints = kagglehub.package_import('metric/svg-constraints')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            try:
                tag_name = etree.QName(element.tag).localname
            except ValueError as e:
                return self.default_svg
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg):
        try:
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
            return base_svg_code
        except Exception as e:
            print(f"Failed to convert {topic} due to {str(e)}, Returning default SVG.")
            return default_svg


class Model:
    def __init__(self):
        self.model="model"
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
        
    def clean_svg(self, base_svg_code: str, max_new_tokens=1024) -> str:
        base_svg_code = SVGProcessor.clean_and_extract_svgs(base_svg_code, self.default_svg)
        #clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return base_svg_code


DEVICE cuda


In [16]:
model=Model()

In [63]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/train_master_description_data.csv',header=[0])
print(df.shape)
df.head(2)

(4188, 1)


,description
0,'Golden wheat fields under a setting sun'
1,'Vibrant orange circles on a cobalt blue backg...


In [18]:
# import warnings
# import logging
# from tqdm import tqdm
# tqdm.pandas()
# df['deepseek_svg'] = df.progress_apply(lambda x: model.clean_svg(x['deepseek_chat']), axis=1)

(100, 1)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 53485.13it/s]


In [64]:
df

,description
0,'Golden wheat fields under a setting sun'
1,'Vibrant orange circles on a cobalt blue backg...
2,'Emerald silk dress with silver embroidery'
3,'Snowy mountains under a clear blue sky'
4,'Checkerboard pattern with alternating green a...
...,...
4183,Abstract representation of fortitude
4184,Pleated willpower
4185,Country awareness
4186,Minimalist courage


In [65]:
df['words'] = df['description'].str.lower().str.split()

In [66]:
all_words = df['words'].explode()

word_counts = all_words.value_counts()

print(word_counts)

words
a           2862
of          1062
with         991
in           619
abstract     543
            ... 
beans          1
hillside       1
pumpkins       1
radishes       1
courage        1
Name: count, Length: 2570, dtype: int64


In [73]:
# Explode the 'words' column into individual rows
all_words = df['words'].explode()

# Drop duplicates to keep only unique words
unique_words = all_words.drop_duplicates().reset_index(drop=True)

print(unique_words)


0         'golden
1           wheat
2          fields
3           under
4               a
          ...    
2566     tenacity
2567         grit
2568    willpower
2569    fortitude
2570      courage
Name: words, Length: 2571, dtype: object


In [74]:
unique_words.tolist()

["'golden",
 'wheat',
 'fields',
 'under',
 'a',
 'setting',
 "sun'",
 "'vibrant",
 'orange',
 'circles',
 'on',
 'cobalt',
 'blue',
 "background'",
 "'emerald",
 'silk',
 'dress',
 'with',
 'silver',
 "embroidery'",
 "'snowy",
 'mountains',
 'clear',
 "sky'",
 "'checkerboard",
 'pattern',
 'alternating',
 'green',
 'and',
 'black',
 "squares'",
 "'turquoise",
 'lagoon',
 'surrounded',
 'by',
 'rocky',
 "cliffs'",
 "'crimson",
 'gold',
 'spirals',
 "intertwining'",
 "'a",
 'navy',
 'trench',
 'coat',
 'brass',
 "buttons'",
 "'scarlet",
 'rectangles',
 'cream',
 "backdrop'",
 "'sunset",
 'over',
 'tranquil',
 'lake',
 'pine',
 "trees'",
 "'geometric",
 'shapes',
 'in',
 'kaleidoscope',
 'of',
 "colors'",
 "'purple",
 'velvet',
 'jacket',
 "piping'",
 "'rolling",
 'hills',
 'patches',
 "wildflowers'",
 "'abstract",
 'swirls',
 'shades',
 'pink',
 "purple'",
 "'denim",
 'jeans',
 'embroidered',
 'floral',
 "patterns'",
 'desert',
 'dunes',
 'starry',
 "'concentric",
 'gray',
 "white'",
 "